# Feature Engineering con Modelos de Hugging Face
### Caso aplicado: Inferencia de sexo desde nombres de candidatos al Congreso de Colombia 2026

---

**Curso:** Analítica de Datos y Machine Learning — Ciencias Políticas, Universidad de Antioquia  
**Objetivo:** Aprender a usar modelos pre-entrenados de Hugging Face como herramienta de **Feature Engineering**, creando nuevas variables (features) a partir de datos no estructurados.

---

### ¿Qué vamos a hacer?

Tenemos una base de datos con **nombres de candidatos** a las elecciones legislativas de Colombia (8 de marzo de 2026). El dataset solo tiene dos columnas: `partido_nombre` y `candidato_nombre`. No tenemos la variable **sexo**.

Vamos a usar **2 modelos complementarios de Hugging Face** para inferir el sexo probable de cada candidato a partir de su nombre:

| Modelo | Tipo | Estrategia |
|---|---|---|
| `padmajabfrl/Gender-Classification` | Clasificador especializado | Rápido, procesa todos los candidatos |
| `google/gemma-4-E2B-it` | Modelo generativo (LLM) | Más lento, se usa solo para casos dudosos |

La idea clave es que **el Modelo 1 hace el trabajo pesado** y **el Modelo 2 actúa como árbitro** para los casos donde el primero no está seguro. Esto es una estrategia común en producción: usar modelos livianos como filtro y modelos pesados como respaldo.

### ¿Por qué es esto Feature Engineering?

Porque estamos **creando nuevas columnas** (features) que no existían en los datos originales, usando modelos de inteligencia artificial como transformadores de información. En la práctica profesional, esto se aplica a:

- Inferir idioma de textos libres
- Clasificar sentimiento de reseñas
- Extraer entidades (nombres, lugares, fechas) de documentos
- Detectar temas en noticias
- Generar embeddings para clustering

### Nota ética importante

Inferir sexo/género desde nombres **no es un problema limpio**. Lo que obtenemos es una **probabilidad de sexo inferido por nombre**, no una verdad observada. Los nombres son culturalmente ambiguos, hay personas trans y no binarias, y un modelo entrenado refleja sesgos de sus datos de entrenamiento. Usamos esta técnica como ejercicio metodológico, conscientes de sus limitaciones.

---
### Flujo general del notebook

```
┌─────────────────────────────────────────────────────────────────────┐
│  DATOS CRUDOS: partido_nombre + candidato_nombre                    │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 1: Extraer primer_nombre                                      │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 2: MODELO 1 (Clasificador) → procesa TODOS los candidatos     │
│          Genera: sexo_m1 + confianza_m1                             │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                  ┌────────┴────────┐
                  │                 │
            confianza ≥ umbral   confianza < umbral
            (casos seguros)      (casos dudosos)
                  │                 │
                  │                 ▼
                  │  ┌────────────────────────────────────────────────┐
                  │  │  PASO 3: MODELO 2 (Gemma) → solo los dudosos  │
                  │  │          Genera: sexo_m2                       │
                  │  └──────────────────┬─────────────────────────────┘
                  │                     │
                  └────────┬────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 4: RECONCILIACIÓN → sexo_final                                │
│          Regla: si M1 seguro → usar M1, si no → usar M2            │
└──────────────────────────┬──────────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────────┐
│  PASO 5: ANÁLISIS EXPLORATORIO con la nueva feature                 │
└─────────────────────────────────────────────────────────────────────┘
```

---
## Paso 1: Instalación de dependencias

Instalamos las librerías necesarias. En Google Colab muchas ya vienen preinstaladas, pero `transformers` y `huggingface_hub` conviene actualizarlas.

In [15]:
!pip install -q transformers huggingface_hub accelerate torch pandas tqdm plotly

In [1]:
!pip install -U -q "bitsandbytes>=0.46.1" accelerate transformers

---
## Paso 2: Autenticación en Hugging Face

El modelo Gemma de Google requiere aceptar una licencia y autenticarse con un token personal.

**Instrucciones para los estudiantes:**
1. Crear una cuenta en [huggingface.co](https://huggingface.co)
2. Ir a [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Crear un token de tipo **Read**
4. En Colab: clic en el ícono de llave 🔑 (panel izquierdo) → agregar un secreto llamado `HF_TOKEN` con tu token
5. Para Gemma: visitar la [página del modelo](https://huggingface.co/google/gemma-3n-E2B-it) y aceptar la licencia

In [2]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get('HF_TOKEN')
login(token)
print("Autenticación exitosa en Hugging Face")

Autenticación exitosa en Hugging Face


---
## Paso 3: Carga y exploración de los datos

In [3]:
import pandas as pd

df = pd.read_csv('congreso_candidatos_2026.csv')

print(f"Total de candidatos: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print(f"Partidos únicos: {df['partido_nombre'].nunique()}")
print("=" * 50)
df.head(10)

Total de candidatos: 2,844
Columnas: ['partido_nombre', 'candidato_nombre']
Partidos únicos: 180


,partido_nombre,candidato_nombre
0,AVANCEMOS NARIÑO,ALEJANDRA GABRIELA ABASOLO GOMEZ
1,AVANCEMOS NARIÑO,EDGAR FERNANDO RAMIREZ BRAVO
2,AVANCEMOS NARIÑO,CARLOS ANDRES CUAICAL CHAPI
3,AVANCEMOS NARIÑO,DEY YAMA CORDOBA
4,AVANCEMOS NARIÑO,ROBERT WILSON SALDAÑA BASANTE
5,PARTIDO LIBERAL COLOMBIANO,JULIO ANIBAL ALVAREZ LOPEZ
6,PARTIDO LIBERAL COLOMBIANO,SERGIO ANTONIO MUÑOZ CASTILLO
7,PARTIDO LIBERAL COLOMBIANO,ALVARO JAVIER ZARAMA BURBANO
8,PARTIDO LIBERAL COLOMBIANO,SONIA ZORAIDA CIFUENTES MELO
9,PARTIDO LIBERAL COLOMBIANO,NATALIA ROMERO VEGA


### Preparación: Extraer el primer nombre

Los modelos de clasificación de género funcionan mejor con el **primer nombre** que con el nombre completo. Los apellidos no aportan señal de género y pueden confundir al modelo.

Creamos una columna auxiliar `primer_nombre` para usarla como input del Modelo 1.

In [4]:
df['primer_nombre'] = df['candidato_nombre'].str.split().str[0]

print("Ejemplos de extracción:")
print("=" * 50)
df[['candidato_nombre', 'primer_nombre']].head(10)

Ejemplos de extracción:


,candidato_nombre,primer_nombre
0,ALEJANDRA GABRIELA ABASOLO GOMEZ,ALEJANDRA
1,EDGAR FERNANDO RAMIREZ BRAVO,EDGAR
2,CARLOS ANDRES CUAICAL CHAPI,CARLOS
3,DEY YAMA CORDOBA,DEY
4,ROBERT WILSON SALDAÑA BASANTE,ROBERT
5,JULIO ANIBAL ALVAREZ LOPEZ,JULIO
6,SERGIO ANTONIO MUÑOZ CASTILLO,SERGIO
7,ALVARO JAVIER ZARAMA BURBANO,ALVARO
8,SONIA ZORAIDA CIFUENTES MELO,SONIA
9,NATALIA ROMERO VEGA,NATALIA


---
## Paso 4: Modelo 1 — `padmajabfrl/Gender-Classification`

📦 **Tipo:** Pipeline de `text-classification` (clasificador especializado)  
🔗 **Enlace:** [padmajabfrl/Gender-Classification](https://huggingface.co/padmajabfrl/Gender-Classification)  
⚡ **Ventaja:** Ligero, rápido, fácil de usar  
⚠️ **Limitación:** Solo devuelve la etiqueta ganadora y su score (no la distribución completa de probabilidades)

### ¿Cómo funciona?

Es un modelo fine-tuneado sobre DistilBERT para clasificar nombres en categorías de género. Se usa a través del pipeline `text-classification` de la librería `transformers`. El pipeline abstrae toda la complejidad: tokenización, inferencia y post-procesamiento en una sola línea.

In [5]:
from transformers import pipeline

# Cargamos el modelo 1: un clasificador especializado
clasificador_m1 = pipeline("text-classification", model="padmajabfrl/Gender-Classification")

print("✅ Modelo 1 cargado exitosamente")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Modelo 1 cargado exitosamente


### Prueba rápida del Modelo 1

Antes de aplicar el modelo a todo el dataset, siempre es buena práctica probarlo con casos conocidos. Esto nos permite verificar que funciona y entender la estructura de su output.

In [6]:
# Prueba rápida con nombres que conocemos
for nombre in ["MARIA", "CARLOS", "ALEJANDRO", "ANDREA", "DEY"]:
    res = clasificador_m1(nombre)[0]
    print(f"  {nombre:15s} → {res['label']:10s} (confianza: {res['score']:.4f})")

  MARIA           → Female     (confianza: 1.0000)
  CARLOS          → Male       (confianza: 1.0000)
  ALEJANDRO       → Male       (confianza: 0.9999)
  ANDREA          → Female     (confianza: 0.9997)
  DEY             → Male       (confianza: 0.7826)


**Observemos:** Cada predicción viene con una etiqueta (`Female`/`Male`) y un score de confianza entre 0.5 y 1.0. Nombres claramente masculinos o femeninos tendrán scores altos. Nombres ambiguos (como DEY) tendrán scores más bajos.

### Aplicar Modelo 1 a todo el dataset

Usamos `tqdm` para tener una barra de progreso. Procesamos en **batches** (lotes) de 64 nombres para mayor eficiencia.

In [7]:
from tqdm.auto import tqdm

# Procesamiento por lotes (batch) para eficiencia
BATCH_SIZE = 64
nombres = df['primer_nombre'].tolist()

resultados_m1 = []
for i in tqdm(range(0, len(nombres), BATCH_SIZE), desc="Modelo 1"):
    batch = nombres[i:i+BATCH_SIZE]
    preds = clasificador_m1(batch)
    resultados_m1.extend(preds)

# Crear las nuevas columnas (features)
df['sexo_m1'] = [r['label'] for r in resultados_m1]
df['confianza_m1'] = [round(r['score'], 4) for r in resultados_m1]

print(f"\nModelo 1 completado para {len(df):,} candidatos")
print(f"\nDistribución de predicciones:")
print(df['sexo_m1'].value_counts())
print(f"\nConfianza promedio: {df['confianza_m1'].mean():.4f}")
print(f"Confianza mínima:  {df['confianza_m1'].min():.4f}")
print(f"Confianza máxima:  {df['confianza_m1'].max():.4f}")

Modelo 1:   0%|          | 0/45 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Modelo 1 completado para 2,844 candidatos

Distribución de predicciones:
sexo_m1
Male      1761
Female    1083
Name: count, dtype: int64

Confianza promedio: 0.9899
Confianza mínima:  0.5236
Confianza máxima:  1.0000


In [8]:
df.head(10)

,partido_nombre,candidato_nombre,primer_nombre,sexo_m1,confianza_m1
0,AVANCEMOS NARIÑO,ALEJANDRA GABRIELA ABASOLO GOMEZ,ALEJANDRA,Male,1.0000
1,AVANCEMOS NARIÑO,EDGAR FERNANDO RAMIREZ BRAVO,EDGAR,Male,1.0000
2,AVANCEMOS NARIÑO,CARLOS ANDRES CUAICAL CHAPI,CARLOS,Male,1.0000
3,AVANCEMOS NARIÑO,DEY YAMA CORDOBA,DEY,Male,0.7826
4,AVANCEMOS NARIÑO,ROBERT WILSON SALDAÑA BASANTE,ROBERT,Male,1.0000
5,PARTIDO LIBERAL COLOMBIANO,JULIO ANIBAL ALVAREZ LOPEZ,JULIO,Male,1.0000
6,PARTIDO LIBERAL COLOMBIANO,SERGIO ANTONIO MUÑOZ CASTILLO,SERGIO,Male,1.0000
7,PARTIDO LIBERAL COLOMBIANO,ALVARO JAVIER ZARAMA BURBANO,ALVARO,Male,1.0000
8,PARTIDO LIBERAL COLOMBIANO,SONIA ZORAIDA CIFUENTES MELO,SONIA,Female,1.0000
9,PARTIDO LIBERAL COLOMBIANO,NATALIA ROMERO VEGA,NATALIA,Female,1.0000


### Inspección: ¿dónde falla o duda el Modelo 1?

Un paso fundamental en Feature Engineering es **no confiar ciegamente en el modelo**. Revisemos los casos donde el clasificador tiene menos confianza — estos son los candidatos donde el nombre es ambiguo o poco frecuente en los datos de entrenamiento.

In [9]:
# Casos con confianza más baja (el modelo "duda")
print("🔍 Top 15 casos con menor confianza (Modelo 1):")
print("=" * 70)
print(df.nsmallest(15, 'confianza_m1')[['candidato_nombre', 'primer_nombre', 'sexo_m1', 'confianza_m1']].to_string(index=False))

🔍 Top 15 casos con menor confianza (Modelo 1):
                candidato_nombre primer_nombre sexo_m1  confianza_m1
FAUSTINO HORACIO HUDGSON WALTERS      FAUSTINO    Male        0.5236
     ARNULFO MOSTACILLA CARABALI       ARNULFO  Female        0.5313
  ARNULFO MANUEL ZAMBRANO MUNIVE       ARNULFO  Female        0.5313
          YULEINY ARIAS MOSQUERA       YULEINY    Male        0.5326
      MELQUISEDEC TEJADA JIMENEZ   MELQUISEDEC  Female        0.5341
   YEIN CAROLINA GUERRERO OSPINA          YEIN    Male        0.5357
           YASMITH VARGAS OYUELA       YASMITH  Female        0.5601
   ALIRIO BENIGNO CUERO QUIÑONEZ        ALIRIO  Female        0.5608
              ALIRIO URIBE MUÑOZ        ALIRIO  Female        0.5608
     MAURO SAUL SANCHEZ ZAMBRANO         MAURO  Female        0.5731
         MAURO JOSE GARCIA GOMEZ         MAURO  Female        0.5731
           GENER ALEXANDER USUGA         GENER  Female        0.5762
           TARSICIO RIVERA MUÑOZ      TARSICIO    Male  

In [10]:
# Distribución de confianza: ¿cuántos casos son "seguros" vs "dudosos"?
import plotly.express as px

fig = px.histogram(
    df, x='confianza_m1', nbins=50,
    title='Distribución de confianza del Modelo 1',
    labels={'confianza_m1': 'Score de confianza', 'count': 'Candidatos'},
    color_discrete_sequence=['#636EFA']
)
fig.add_vline(x=0.95, line_dash="dash", line_color="red",
              annotation_text="Umbral: 0.95", annotation_position="top left")
fig.update_layout(template='plotly_white')
fig.show()

In [11]:
# Definimos un umbral de confianza
UMBRAL_CONFIANZA = 0.95

seguros = df[df['confianza_m1'] >= UMBRAL_CONFIANZA]
dudosos = df[df['confianza_m1'] < UMBRAL_CONFIANZA]

print(f"Candidatos con confianza ≥ {UMBRAL_CONFIANZA}: {len(seguros):,} ({len(seguros)/len(df)*100:.1f}%)")
print(f"Candidatos con confianza <  {UMBRAL_CONFIANZA}: {len(dudosos):,} ({len(dudosos)/len(df)*100:.1f}%)")
print(f"\n→ El Modelo 2 (Gemma) solo procesará los {len(dudosos):,} casos dudosos.")

Candidatos con confianza ≥ 0.95: 2,722 (95.7%)
Candidatos con confianza <  0.95: 122 (4.3%)

→ El Modelo 2 (Gemma) solo procesará los 122 casos dudosos.


---
## Paso 5: Modelo 2 — `google/gemma-4-E2B-it` (Modelo Generativo)

📦 **Tipo:** Modelo generativo (LLM) con prompt engineering  
🔗 **Enlace:** [google/gemma-3n-E2B-it](https://huggingface.co/google/gemma-3n-E2B-it)  
⚡ **Ventaja:** Entiende contexto cultural colombiano, puede manejar nombres ambiguos con razonamiento  
⚠️ **Limitación:** Más lento, más costoso en GPU, no da probabilidades calibradas

### ¿Cuál es la diferencia fundamental?

El Modelo 1 es un **clasificador especializado**: fue entrenado específicamente para clasificar género desde nombres. Gemma es un **modelo de lenguaje general** al que le pedimos que clasifique mediante un prompt (instrucciones en texto). Esto se llama **prompt engineering** y es una estrategia de Feature Engineering cada vez más usada.

| Aspecto | Modelo 1 (Clasificador) | Modelo 2 (Gemma LLM) |
|---|---|---|
| Entrenamiento | Fine-tuned para esta tarea | Modelo general de lenguaje |
| Input | Solo primer nombre | Nombre completo (más contexto) |
| Output | Etiqueta + probabilidad calibrada | Texto generado (M/F/I) |
| Velocidad | ~45 nombres/segundo | ~1-2 nombres/segundo |
| Uso en este notebook | Todos los candidatos | Solo los dudosos del M1 |

**⚠️ Requisito:** Este modelo requiere GPU. En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

**⚠️ Licencia:** Debes aceptar la licencia del modelo en su [página de Hugging Face](https://huggingface.co/google/gemma-3n-E2B-it).

In [12]:
import torch
from transformers import AutoProcessor, Gemma3nForConditionalGeneration

# Verificar disponibilidad de GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No hay GPU disponible. Gemma será muy lento en CPU.")
    print("   Ve a: Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)")

Dispositivo: cuda
GPU: Tesla T4
Memoria GPU: 15.6 GB


In [13]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Modelo Gemma cargado exitosamente en GPU")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Modelo Gemma cargado exitosamente en GPU


### La función de clasificación con Gemma

Aquí es donde aplicamos **prompt engineering**: le damos al modelo instrucciones precisas de cómo queremos que responda. El prompt le dice que es un clasificador de nombres colombianos y que debe responder con una sola letra.

Para Gemma usamos el **nombre completo** (no solo el primer nombre), ya que al ser un LLM puede aprovechar el contexto cultural de nombres y apellidos colombianos.

In [14]:
def clasificar_sexo_gemma(nombre: str) -> dict:
    """
    Clasifica el sexo probable de un nombre colombiano usando Gemma.
    Retorna un diccionario con la etiqueta predicha (M/F/I).
    """
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Eres un clasificador de nombres colombianos. "
                        "Debes responder con una sola letra y nada más. "
                        "M = masculino probable. "
                        "F = femenino probable. "
                        "I = indeterminado o ambiguo."
                    )
                }
            ]
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"Nombre completo: {nombre}"
                }
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False
        )

    texto = processor.batch_decode(
        output,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0].strip().upper()

    # Extraer la etiqueta de los últimos caracteres generados
    if "F" in texto[-3:]:
        etiqueta = "F"
    elif "M" in texto[-3:]:
        etiqueta = "M"
    else:
        etiqueta = "I"

    return {"label": etiqueta, "raw_response": texto[-10:]}

### Prueba rápida del Modelo 2

In [15]:
# Prueba rápida con nombres completos
for nombre in ["MARIA ALEJANDRA PATIÑO DAZA", "CARLOS ANDRES CUAICAL CHAPI", "DEY YAMA CORDOBA"]:
    res = clasificar_sexo_gemma(nombre)
    print(f"  {nombre:35s} → {res['label']}  (raw: {res['raw_response']})")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  MARIA ALEJANDRA PATIÑO DAZA         → F  (raw: ZA
MODEL
F)
  CARLOS ANDRES CUAICAL CHAPI         → M  (raw: PI
MODEL
M)
  DEY YAMA CORDOBA                    → M  (raw: BA
MODEL
M)


### Aplicar Modelo 2 solo a los casos dudosos

Esta es la estrategia clave del notebook: **no procesamos todos los 2,844 candidatos con Gemma** (lo cual tomaría mucho tiempo), sino solo aquellos donde el Modelo 1 no está seguro.

**⏱️ Tiempo estimado:** Depende de la cantidad de casos dudosos y la GPU asignada. Con T4, cada nombre toma ~0.5-1 segundo.

In [16]:
# 1. Identificamos los candidatos dudosos
UMBRAL_CONFIANZA = 0.95
mask_dudosos = df['confianza_m1'] < UMBRAL_CONFIANZA
df_dudosos = df[mask_dudosos].copy()

print(f"Candidatos a procesar con Modelo 2 (Gemma): {len(df_dudosos):,}")
print(f"Esto es solo el {len(df_dudosos)/len(df)*100:.1f}% del total → ahorro significativo de tiempo")

Candidatos a procesar con Modelo 2 (Gemma): 122
Esto es solo el 4.3% del total → ahorro significativo de tiempo


In [17]:
# 2. Inicializamos la columna sexo_m2 como NaN para todos
# Solo los dudosos tendrán un valor del Modelo 2
df['sexo_m2'] = pd.NA

# 3. Ejecutamos Gemma sobre los casos dudosos
if not df_dudosos.empty:
    nombres_dudosos = df_dudosos['candidato_nombre'].tolist()
    resultados_m2 = []

    for nombre in tqdm(nombres_dudosos, desc="Modelo 2 (Gemma)"):
        try:
            res = clasificar_sexo_gemma(nombre)
            resultados_m2.append(res['label'])
        except Exception as e:
            print(f"  ⚠️ Error con '{nombre}': {e}")
            resultados_m2.append("I")

    # Asignar resultados solo a las filas dudosas
    df.loc[mask_dudosos, 'sexo_m2'] = resultados_m2

print("\nProcesamiento con Modelo 2 completado.")
print(f"\nDistribución de predicciones del Modelo 2 (solo casos dudosos):")
print(df.loc[mask_dudosos, 'sexo_m2'].value_counts())

Modelo 2 (Gemma):   0%|          | 0/122 [00:00<?, ?it/s]


Procesamiento con Modelo 2 completado.

Distribución de predicciones del Modelo 2 (solo casos dudosos):
sexo_m2
M    90
F    32
Name: count, dtype: int64


---
## Paso 6: Reconciliación — Crear la feature final `sexo_final`

Ahora viene el paso más importante del Feature Engineering: **combinar los resultados de ambos modelos** en una sola columna definitiva.

La regla de decisión es simple:

- **Si el Modelo 1 tiene confianza ≥ 0.95** → usamos la predicción del Modelo 1
- **Si el Modelo 1 tiene confianza < 0.95** → usamos la predicción del Modelo 2 (Gemma)

También creamos una columna `fuente_prediccion` para saber de dónde vino cada clasificación. Esto es una buena práctica de trazabilidad.

In [18]:
# Homologar etiquetas del Modelo 1 al formato M/F
# Modelo 1 usa "Female"/"Male", Modelo 2 usa "F"/"M"
df['sexo_m1_homologado'] = df['sexo_m1'].map({'Female': 'F', 'Male': 'M'})

# Crear la feature final con la regla de decisión
df['sexo_final'] = df['sexo_m1_homologado']  # Por defecto, usamos M1
df['fuente_prediccion'] = 'Modelo 1'          # Por defecto, la fuente es M1

# Para los casos dudosos, sobreescribimos con el resultado de Gemma
mask_dudosos = df['confianza_m1'] < UMBRAL_CONFIANZA
df.loc[mask_dudosos, 'sexo_final'] = df.loc[mask_dudosos, 'sexo_m2']
df.loc[mask_dudosos, 'fuente_prediccion'] = 'Modelo 2'

# Resumen
print("✅ Feature 'sexo_final' creada exitosamente")
print("=" * 50)
print(f"\nDistribución final:")
print(df['sexo_final'].value_counts())
print(f"\nFuente de cada predicción:")
print(df['fuente_prediccion'].value_counts())

✅ Feature 'sexo_final' creada exitosamente

Distribución final:
sexo_final
M    1759
F    1085
Name: count, dtype: int64

Fuente de cada predicción:
fuente_prediccion
Modelo 1    2722
Modelo 2     122
Name: count, dtype: int64


In [19]:
# Vista del dataset con todas las features creadas
columnas_resumen = ['partido_nombre', 'candidato_nombre', 'primer_nombre',
                    'sexo_m1', 'confianza_m1', 'sexo_m2', 'sexo_final', 'fuente_prediccion']
df[columnas_resumen].head(15)

,partido_nombre,candidato_nombre,primer_nombre,sexo_m1,confianza_m1,sexo_m2,sexo_final,fuente_prediccion
0,AVANCEMOS NARIÑO,ALEJANDRA GABRIELA ABASOLO GOMEZ,ALEJANDRA,Male,1.0000,<NA>,M,Modelo 1
1,AVANCEMOS NARIÑO,EDGAR FERNANDO RAMIREZ BRAVO,EDGAR,Male,1.0000,<NA>,M,Modelo 1
2,AVANCEMOS NARIÑO,CARLOS ANDRES CUAICAL CHAPI,CARLOS,Male,1.0000,<NA>,M,Modelo 1
3,AVANCEMOS NARIÑO,DEY YAMA CORDOBA,DEY,Male,0.7826,M,M,Modelo 2
4,AVANCEMOS NARIÑO,ROBERT WILSON SALDAÑA BASANTE,ROBERT,Male,1.0000,<NA>,M,Modelo 1
5,PARTIDO LIBERAL COLOMBIANO,JULIO ANIBAL ALVAREZ LOPEZ,JULIO,Male,1.0000,<NA>,M,Modelo 1
6,PARTIDO LIBERAL COLOMBIANO,SERGIO ANTONIO MUÑOZ CASTILLO,SERGIO,Male,1.0000,<NA>,M,Modelo 1
7,PARTIDO LIBERAL COLOMBIANO,ALVARO JAVIER ZARAMA BURBANO,ALVARO,Male,1.0000,<NA>,M,Modelo 1
8,PARTIDO LIBERAL COLOMBIANO,SONIA ZORAIDA CIFUENTES MELO,SONIA,Female,1.0000,<NA>,F,Modelo 1
9,PARTIDO LIBERAL COLOMBIANO,NATALIA ROMERO VEGA,NATALIA,Female,1.0000,<NA>,F,Modelo 1


### Verificación: ¿Los modelos coinciden en los casos dudosos?

Veamos en cuántos casos los dos modelos están de acuerdo y en cuántos difieren. Esto nos da una idea de la robustez de nuestras predicciones.

In [20]:
# Solo para los casos donde ambos modelos opinaron
df_comparacion = df[mask_dudosos].copy()
df_comparacion['coinciden'] = df_comparacion['sexo_m1_homologado'] == df_comparacion['sexo_m2']

n_coinciden = df_comparacion['coinciden'].sum()
n_difieren = len(df_comparacion) - n_coinciden

print(f"Casos procesados por ambos modelos: {len(df_comparacion):,}")
print(f"  ✅ Coinciden: {n_coinciden:,} ({n_coinciden/len(df_comparacion)*100:.1f}%)")
print(f"  ❌ Difieren:  {n_difieren:,} ({n_difieren/len(df_comparacion)*100:.1f}%)")

if n_difieren > 0:
    print(f"\n🔍 Casos donde los modelos no coinciden:")
    print("=" * 70)
    cols = ['candidato_nombre', 'primer_nombre', 'sexo_m1', 'confianza_m1', 'sexo_m2']
    print(df_comparacion[~df_comparacion['coinciden']][cols].head(20).to_string(index=False))

Casos procesados por ambos modelos: 122
  ✅ Coinciden: 80 (65.6%)
  ❌ Difieren:  42 (34.4%)

🔍 Casos donde los modelos no coinciden:
                  candidato_nombre primer_nombre sexo_m1  confianza_m1 sexo_m2
        YARLEIDIS MURILLO PALACIOS     YARLEIDIS    Male        0.9351       F
          ROMELIO RIASCOS RENTERIA       ROMELIO  Female        0.8688       M
       MAURO SAUL SANCHEZ ZAMBRANO         MAURO  Female        0.5731       M
        MELQUISEDEC TEJADA JIMENEZ   MELQUISEDEC  Female        0.5341       M
KARELIS CAROLINA CARABALLO SALINAS       KARELIS    Male        0.6867       F
             ROLANDO GARCIA ALFARO       ROLANDO  Female        0.9171       M
     JULY ESPERANZA GONZALEZ GOMEZ          JULY    Male        0.5926       F
        JHAN ELER PEÑALOZA DELGADO          JHAN  Female        0.7012       M
       DANILO JOSE DE ARMAS IPUANA        DANILO  Female        0.8328       M
     KELLY JULIANA NAVAS HERNANDEZ         KELLY    Male        0.9219       

---
## Paso 7: Análisis exploratorio con la nueva feature

Ahora que tenemos la variable `sexo_final`, podemos hacer el tipo de análisis que motiva todo este ejercicio: **entender la composición de género en las listas de candidatos al Congreso**.

Este es el valor real del Feature Engineering: convertir datos crudos (nombres) en variables analíticas útiles.

In [21]:
# Distribución general de sexo inferido
fig = px.pie(
    df, names='sexo_final',
    title='Distribución de sexo inferido — Candidatos al Congreso 2026',
    color_discrete_map={'F': '#EF553B', 'M': '#636EFA', 'I': '#AB63FA'},
    hole=0.4
)
fig.update_traces(textinfo='label+percent+value')
fig.update_layout(template='plotly_white')
fig.show()

In [22]:
# Top 20 partidos con más candidatas mujeres (proporción)
partido_genero = df.groupby('partido_nombre')['sexo_final'].value_counts(normalize=True).unstack(fill_value=0)

# Solo partidos con al menos 5 candidatos para que la proporción sea significativa
partidos_con_datos = df['partido_nombre'].value_counts()
partidos_validos = partidos_con_datos[partidos_con_datos >= 5].index
partido_genero_filtrado = partido_genero.loc[partido_genero.index.isin(partidos_validos)]

if 'F' in partido_genero_filtrado.columns:
    top_paridad = partido_genero_filtrado.sort_values('F', ascending=False).head(20)

    fig = px.bar(
        top_paridad.reset_index(),
        x='F', y='partido_nombre',
        orientation='h',
        title='Top 20 partidos con mayor proporción de candidatas mujeres<br><sup>(Solo partidos con ≥5 candidatos)</sup>',
        labels={'F': 'Proporción de mujeres', 'partido_nombre': ''},
        color_discrete_sequence=['#EF553B']
    )
    fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'}, height=600)
    fig.show()

In [23]:
# Confianza del Modelo 1 por sexo predicho
fig = px.box(
    df, x='sexo_m1', y='confianza_m1',
    title='Distribución de confianza del Modelo 1 por categoría predicha',
    labels={'sexo_m1': 'Sexo predicho (M1)', 'confianza_m1': 'Confianza'},
    color='sexo_m1',
    color_discrete_map={'Female': '#EF553B', 'Male': '#636EFA'}
)
fig.update_layout(template='plotly_white')
fig.show()

---
## Paso 8: Exportar el dataset enriquecido

Guardamos el dataset con todas las features creadas. Este archivo es el producto final de nuestro proceso de Feature Engineering y puede ser usado para análisis posteriores o como input de modelos de Machine Learning.

In [24]:
# Seleccionar las columnas finales relevantes
df_export = df[['partido_nombre', 'candidato_nombre', 'primer_nombre',
                'sexo_m1', 'confianza_m1', 'sexo_m2',
                'sexo_final', 'fuente_prediccion']].copy()

# Exportar
df_export.to_csv('candidatos_congreso_2026_con_features.csv', index=False)

print(f"Dataset exportado: candidatos_congreso_2026_con_features.csv")
print(f"   Filas: {len(df_export):,}")
print(f"   Columnas: {len(df_export.columns)}")
print(f"\nColumnas del dataset final:")
for col in df_export.columns:
    print(f"   • {col}")

Dataset exportado: candidatos_congreso_2026_con_features.csv
   Filas: 2,844
   Columnas: 8

Columnas del dataset final:
   • partido_nombre
   • candidato_nombre
   • primer_nombre
   • sexo_m1
   • confianza_m1
   • sexo_m2
   • sexo_final
   • fuente_prediccion


In [25]:
df_export.head(10)

,partido_nombre,candidato_nombre,primer_nombre,sexo_m1,confianza_m1,sexo_m2,sexo_final,fuente_prediccion
0,AVANCEMOS NARIÑO,ALEJANDRA GABRIELA ABASOLO GOMEZ,ALEJANDRA,Male,1.0000,<NA>,M,Modelo 1
1,AVANCEMOS NARIÑO,EDGAR FERNANDO RAMIREZ BRAVO,EDGAR,Male,1.0000,<NA>,M,Modelo 1
2,AVANCEMOS NARIÑO,CARLOS ANDRES CUAICAL CHAPI,CARLOS,Male,1.0000,<NA>,M,Modelo 1
3,AVANCEMOS NARIÑO,DEY YAMA CORDOBA,DEY,Male,0.7826,M,M,Modelo 2
4,AVANCEMOS NARIÑO,ROBERT WILSON SALDAÑA BASANTE,ROBERT,Male,1.0000,<NA>,M,Modelo 1
5,PARTIDO LIBERAL COLOMBIANO,JULIO ANIBAL ALVAREZ LOPEZ,JULIO,Male,1.0000,<NA>,M,Modelo 1
6,PARTIDO LIBERAL COLOMBIANO,SERGIO ANTONIO MUÑOZ CASTILLO,SERGIO,Male,1.0000,<NA>,M,Modelo 1
7,PARTIDO LIBERAL COLOMBIANO,ALVARO JAVIER ZARAMA BURBANO,ALVARO,Male,1.0000,<NA>,M,Modelo 1
8,PARTIDO LIBERAL COLOMBIANO,SONIA ZORAIDA CIFUENTES MELO,SONIA,Female,1.0000,<NA>,F,Modelo 1
9,PARTIDO LIBERAL COLOMBIANO,NATALIA ROMERO VEGA,NATALIA,Female,1.0000,<NA>,F,Modelo 1


---
## Resumen y reflexiones finales

### ¿Qué aprendimos?

**1. Feature Engineering con modelos pre-entrenados:** Transformamos una columna de texto (nombres) en una variable categórica analítica (sexo inferido) usando modelos de Hugging Face. Esto es una técnica cada vez más común en ciencia de datos.

**2. Estrategia de modelos complementarios:** Usamos un modelo rápido (clasificador) para el grueso del trabajo y un modelo pesado (LLM) solo para los casos difíciles. Esta estrategia ahorra tiempo y recursos computacionales sin sacrificar calidad.

**3. Trazabilidad:** Guardamos no solo la predicción final, sino también las predicciones individuales de cada modelo, las confianzas y la fuente de cada decisión. Esto permite auditar y mejorar el proceso.

### Tabla resumen de los modelos

| Aspecto | Modelo 1 | Modelo 2 |
|---|---|---|
| Nombre | `padmajabfrl/Gender-Classification` | `google/gemma-3n-E2B-it` |
| Tipo | Clasificador fine-tuned | LLM generativo |
| Candidatos procesados | Todos (2,844) | Solo los dudosos |
| Velocidad | Rápido (batches de 64) | Lento (uno por uno) |
| Output | Etiqueta + score calibrado | Solo etiqueta (M/F/I) |
| Requiere GPU | No necesariamente | Sí (recomendado T4+) |
| Requiere prompt | No | Sí (prompt engineering) |

### Limitaciones y consideraciones éticas

- La inferencia de género desde nombres es una **aproximación estadística**, no una verdad.
- Los modelos reflejan los sesgos de sus datos de entrenamiento (mayoritariamente nombres anglosajones).
- Nombres de origen indígena, afrodescendiente o de culturas no occidentales pueden tener mayor tasa de error.
- Esta técnica es útil para **análisis agregados** (ej: proporción de mujeres por partido), pero no debe usarse para tomar decisiones individuales sobre personas.

### Para seguir explorando

- ¿Qué pasa si cambias el umbral de confianza? Prueba con 0.90 o 0.99.
- ¿Podrías usar un tercer modelo como desempate cuando M1 y M2 difieren?
- ¿Cómo se comparan estos resultados con los datos oficiales del CNE sobre paridad de género?
- Intenta aplicar esta misma lógica a otro problema: clasificar sentimiento de tweets sobre los candidatos.